# Conditioning screen, probe protocol: base

Flags beyond the base: `(none: the control)`. Seeds 0, 1, 2, K=32 / 400k jets / 20 epochs, T4 with
torch 2.11.0+cu128 pinned, paired seed by seed with the base control of the
same screen. Kernel budget 1.55 h.

In [ ]:
import subprocess, time
T0 = time.time()
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True,
                     text=True).stdout.strip())

In [ ]:
import subprocess, sys, time
WANT = "2.11.0+cu128"
name = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                      capture_output=True, text=True).stdout.strip()
print("device:", name)
assert "T4" in name, "paired with T4 runs only; got %r -- stopping before any GPU time is spent" % name

def torch_version():
    return subprocess.run([sys.executable, "-c", "import torch; print(torch.__version__)"],
                          capture_output=True, text=True).stdout.strip()

v = torch_version()
if v != WANT:
    print("torch", v, "- installing", WANT)
    t0 = time.perf_counter()
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.11.0",
                        "--index-url", "https://download.pytorch.org/whl/cu128"],
                       capture_output=True, text=True)
    print("pip rc", r.returncode, "in %.0f s" % (time.perf_counter() - t0))
    if r.returncode:
        print(r.stderr[-2000:])
        raise SystemExit("could not install torch " + WANT)
    v = torch_version()
assert v == WANT, "torch %s, expected %s" % (v, WANT)
print("torch", v, "on", name)
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "torchdiffeq"], check=True)

In [ ]:
import base64, io, tarfile, pathlib
CODE_B64 = "..."
tarfile.open(fileobj=io.BytesIO(base64.b64decode(CODE_B64)), mode="r:gz").extractall("/kaggle/working/repo")
print(sorted(p.name for p in pathlib.Path("/kaggle/working/repo").iterdir()))

In [ ]:
import subprocess, sys

def run(args, cwd="/kaggle/working/repo"):
    """Run a repo module, stream nothing, return (ok, tail-of-output)."""
    r = subprocess.run([sys.executable, "-m"] + args, cwd=cwd,
                       capture_output=True, text=True)
    out = r.stdout[-4000:]
    if r.returncode != 0:
        out += "\n--- stderr ---\n" + r.stderr[-3000:]
    print(out)
    return r.returncode == 0

In [ ]:
import glob, pathlib
cands = glob.glob("/kaggle/input/**/top_tagging_train.npz", recursive=True)
assert cands, "attach the K=128 data-prep kernel output"
DATA = str(pathlib.Path(cands[0]).parent)
TAG, SEEDS = 'base', [0, 1, 2]
OUT = "/kaggle/working/results_probe_screen/" + TAG
BASE = ['--cache-dir', 'DATA', '--representation', 'constituents', '--canonical-splits', '--normalize', 'global', '--max-train-samples', '400000', '--epochs', '20', '--device', 'cuda', '--dtype', 'float32', '--batch-size', '256', '--n-constituents', '32', '--eval-chunk-size', '2048', '--models', 'so3c_message_set', '--rounds', '3', '--beams', '--channels', '8', '--compile']
BASE[BASE.index("DATA")] = DATA
BUDGET = 1.55 * 3600           # this kernel's share of the week's GPU quota
EVAL_MARGIN = 240            # final test evaluation and writing results
MIN_SEED = 1500              # a seed needs ~0.45 h; less than this cannot finish
pathlib.Path(OUT).mkdir(parents=True, exist_ok=True)
pathlib.Path(OUT, "bundle_sha256.txt").write_text('435ec64f997c' + chr(10))
print("data:", DATA, "| bundle", '435ec64f997c')

In [ ]:
# On-card guard: the forward pass must be the published function here too.
import subprocess, sys
g = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider",
                    "tests/test_so3c_models.py", "-k", "golden_cuda"],
                   cwd="/kaggle/working/repo", capture_output=True, text=True)
print(g.stdout[-1200:], g.stderr[-800:])
assert "1 passed" in g.stdout, "golden guard did not pass on the card"

In [ ]:
import glob, time
for seed in SEEDS:
    left = BUDGET - (time.time() - T0) - EVAL_MARGIN
    print("=== seed %d: %.2f h of kernel budget left ===" % (seed, left / 3600), flush=True)
    if left < MIN_SEED:
        print("not enough budget left; seed %d skipped" % seed)
        continue
    run(["benchmarks.run_top_tagging"] + BASE +
        ["--seed", str(seed), "--results-dir", OUT + "/seed%d" % seed,
         "--ckpt-dir", "/kaggle/working/checkpoints/%s_seed%d" % (TAG, seed),
         "--resume", "--max-seconds", str(int(left))])
    assert glob.glob(OUT + "/seed%d/*.json" % seed), "seed %d wrote no result" % seed
print("kernel time %.2f h" % ((time.time() - T0) / 3600))

In [ ]:
import json, glob, statistics
aucs, rejs = [], []
print("%-6s%9s%10s%10s%8s%8s%20s%10s" % ("seed", "AUC", "rej@0.3", "rej@0.5", "epochs", "hours", "torch", "gpu"))
for seed in SEEDS:
    for f in sorted(glob.glob(OUT + "/seed%d/*.json" % seed)):
        r = json.load(open(f)); t = r["test_metrics"]; env = r.get("env", {})
        print("%-6d%9.5f%10.1f%10.1f%8d%8.2f%20s%10s"
              % (seed, t["test_auc"], t["bg_rej_30"], t.get("bg_rej_50", float("nan")),
                 r["epochs_run"], r["walltime_sec"] / 3600, env.get("torch"),
                 env.get("gpu", "").replace("Tesla ", "")))
        assert env.get("torch") == "2.11.0+cu128", "result recorded torch %r" % env.get("torch")
        if r["epochs_run"] < 20:
            print("  INCOMPLETE: stopped at epoch %d on the kernel budget" % r["epochs_run"])
        else:
            aucs.append(t["test_auc"]); rejs.append(t["bg_rej_30"])
if len(aucs) > 1:
    print("complete seeds: mean %.5f sd %.5f AUC | mean %.1f sd %.1f rejection"
          % (statistics.mean(aucs), statistics.stdev(aucs),
             statistics.mean(rejs), statistics.stdev(rejs)))